This is a record of all the code used to compute summary stats for chapter 2.

In [1]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit, prange
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib as mpl
import pandas as pd
from sklearn.multiclass import OneVsRestClassifier as classifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import MultiLabelBinarizer, LabelBinarizer
from sklearn.metrics import log_loss, roc_curve
import optuna
from sklearn.cluster import AgglomerativeClustering as clustering

base_filepath = "/Users/44749/Documents/PhD_work/chapter_2_sims/"

In [5]:

#choose samples, making sure they're not too close together
@njit()
def choose_samples(sample_locs, no_samples):
    chosen_indices, chosen_locs = [0], [sample_locs[0]] #take the first sample always, assume a point chosen at random
    #of the available samples, take the one that maximises the total distance from all current samples
    while len(chosen_indices) < no_samples:
        furthest_current_index, max_total_distance = 0, 0
        for num, loc in enumerate(sample_locs):
            if num not in chosen_indices:
                total_distance = sum([np.sqrt((loc[0]-x)**2 + (loc[1]-y)**2 + (loc[2]-z)**2) for [x, y, z] in chosen_locs])
                if total_distance > max_total_distance:
                    max_total_distance = total_distance
                    furthest_current_index = num
        chosen_indices.append(furthest_current_index)
        chosen_locs.append(sample_locs[furthest_current_index])
    return chosen_indices

#calculate the number of mutations fixed in n samples
@njit()
def count_fixation_number(mut_matrix, num_samples):
    number_fixed = 0
    for row in mut_matrix:
        num_samples_fixed = 0
        for read in row:
            if read == 1.0:
                num_samples_fixed += 1
        if num_samples_fixed == num_samples:
            number_fixed += 1
    return number_fixed


prim_read_threshold = 10/160
second_read_threshold = 4/160

#calculate output summary statistics for a given number of samples, excluding clonal mutations
def calculate_output_stats(filename, base_filepath, number, no_samples):
    #load files
    mutdict = np.load(base_filepath+filename+"/exp_"+str(number)+"_mutdict.npy", allow_pickle=True)
    sample_locs = np.load(base_filepath+filename+"/exp_"+str(number)+"_sample_locs.npy", allow_pickle=True)
    #to mimic actual sampling procedure
    chosen_indices = choose_samples(sample_locs, no_samples)
    sample_dicts = [mutdict[index] for index in chosen_indices]
    #look at all mutations seen
    all_muts = []
    for dict in sample_dicts:
        all_muts += list(dict.keys()) #look at all mutations seen in this sample
    all_unique_mutations = np.unique(all_muts) #all mutations in all samples
    mut_matrix = np.zeros((len(all_unique_mutations), no_samples))
    for m, mut in enumerate(all_unique_mutations):
        for n, dict in enumerate(sample_dicts):
            if mut in dict:
                mut_matrix[m][n] = dict[mut] #zero by default
    max_ccfs = np.max(mut_matrix, axis=1)
    sufficient = np.where(max_ccfs >= prim_read_threshold)[0]
    mut_matrix = mut_matrix[sufficient]
    #put in the read threshold- then compute clonal muts-- if we're not computing clonal muts this order doesn't matter 
    #because everything failing it is already nonclonal
    min_ccfs = np.min(mut_matrix, axis=1)
    num_clonal_muts = len(np.where(min_ccfs == 1.0)[0])
    non_clonal_muts = np.where(min_ccfs < 1.0)[0] #not clonal everywhere
    mut_matrix = mut_matrix[non_clonal_muts] #filter out all of the clonal ones
    #we need it to be detected above 10 reads somewhere; 4 reads in the relevant sample is already built in
    #now to actually calculate summary statistics!
    #0. all mutations detected subclonally (i.e. with CCF < 1.0 in at least one sample)
    no_muts = len(mut_matrix)
    #1. all mutations detected in 1 sample only
    appearances = np.count_nonzero(mut_matrix, axis=1)
    private = np.where(appearances == 1)[0]
    num_private = len(private)
    #2-no_samples, ... number fixed in 1, ... no_samples-1 samples
    fixed_muts = []
    for s in range(1, no_samples):
        fixed_muts.append(count_fixation_number(mut_matrix, s)) #count number fixed in s samples- here it MAY THEN APPEAR IN A NUMBER OF OTHER SAMPLES, JUST NOT FIXED
    #no_samples+1: number of mutations fixed AND private
    private_ccfs = np.max(mut_matrix[private], axis=1) #the maximum CCF of a private mutation will be its CCF in that sample
    fixed_and_private = np.where(private_ccfs == 1.0)[0]
    num_fixed_private = len(fixed_and_private)
    #no_samples + 2-2*no_samples-1- the number of mutations appearing in 2, ... no_samples samples
    num_appearing = []
    for s in range(2, no_samples+1):
        num_appearing.append(len(np.where(appearances == s)[0]))
    return [no_muts, num_private] + fixed_muts + [num_fixed_private] + num_appearing + [num_clonal_muts]   
    
    

In [6]:

#generate parameters for this model-- copy of august_curtis_params

max_sel = 0.1
gran=100
repeats = 20 #bear in mind each of these will be repeated, in simulations, 5 times-- so if we want 100 repeats we only need 20 here

base_sel = np.linspace(0, 0.1, num=gran+1)

selections = []

for sel in base_sel:
    for r in range(repeats):
        selections.append(sel)
    sel += gran

#remember each is repeated for 5 simulations within that-- because of the batching

In [7]:
filenames = ["11Aug_no_death", "11Aug_low_death", "11Aug_mid_death", "11Aug_high_death", "11Aug_vhigh_death", "11Aug_highest_death"]
death_rates = [0.0, 0.1, 0.2, 0.3, 0.35, 0.38]
matrix_names = ["model_1/"+filename for filename in filenames]


batches = len(selections)
batch_size = 5

for (filename, death_rate, matrix_name) in zip(filenames, death_rates, matrix_names):
    for no_samples in range(2, 9):
        print("For "+str(no_samples) + " samples", filename)
        stat_matrix = []
        params_per_sim = []
        for batch in range(batches):
            selection_level = selections[batch] #selection per batch
            for number in range(batch_size*batch+1, batch_size*(batch+1)+1): 
                try:
                    stats = calculate_output_stats(matrix_name, base_filepath, number, no_samples)
                    stat_matrix.append(stats)
                    params_per_sim.append([selection_level, death_rate])
                    #print(stats)
                except:
                    print(filename, matrix_name, number, "failed")
            #print(batch, " complete")
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_stats.npy", stat_matrix)
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_params.npy", params_per_sim)
        

For 2 samples 11Aug_no_death
For 3 samples 11Aug_no_death
For 4 samples 11Aug_no_death
For 5 samples 11Aug_no_death
For 6 samples 11Aug_no_death
For 7 samples 11Aug_no_death
For 8 samples 11Aug_no_death
For 2 samples 11Aug_low_death
For 3 samples 11Aug_low_death
For 4 samples 11Aug_low_death
For 5 samples 11Aug_low_death
For 6 samples 11Aug_low_death
For 7 samples 11Aug_low_death
For 8 samples 11Aug_low_death
For 2 samples 11Aug_mid_death
For 3 samples 11Aug_mid_death
For 4 samples 11Aug_mid_death
For 5 samples 11Aug_mid_death
For 6 samples 11Aug_mid_death
For 7 samples 11Aug_mid_death
For 8 samples 11Aug_mid_death
For 2 samples 11Aug_high_death
For 3 samples 11Aug_high_death
For 4 samples 11Aug_high_death
For 5 samples 11Aug_high_death
For 6 samples 11Aug_high_death
For 7 samples 11Aug_high_death
For 8 samples 11Aug_high_death
For 2 samples 11Aug_vhigh_death
For 3 samples 11Aug_vhigh_death
For 4 samples 11Aug_vhigh_death
For 5 samples 11Aug_vhigh_death
For 6 samples 11Aug_vhigh_death


In [8]:
filenames = ["14Aug_no_death", "14Aug_low_death", "14Aug_mid_death", "14Aug_high_death", "14Aug_vhigh_death", "14Aug_highest_death"]
death_rates = [0.0, 0.1, 0.2, 0.3, 0.35, 0.38]
matrix_names = ["model_2/"+filename for filename in filenames]


batches = len(selections)
batch_size = 5

for (filename, death_rate, matrix_name) in zip(filenames, death_rates, matrix_names):
    for no_samples in range(2, 9):
        print("For "+str(no_samples) + " samples", filename)
        stat_matrix = []
        params_per_sim = []
        for batch in range(batches):
            selection_level = selections[batch] #selection per batch
            for number in range(batch_size*batch+1, batch_size*(batch+1)+1): 
                try:
                    stats = calculate_output_stats(matrix_name, base_filepath, number, no_samples)
                    stat_matrix.append(stats)
                    params_per_sim.append([selection_level, death_rate])
                    #print(stats)
                except:
                    print(filename, matrix_name, number, "failed")
            #print(batch, " complete")
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_stats.npy", stat_matrix)
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_params.npy", params_per_sim)
        

For 2 samples 14Aug_no_death
For 3 samples 14Aug_no_death
For 4 samples 14Aug_no_death
For 5 samples 14Aug_no_death
For 6 samples 14Aug_no_death
For 7 samples 14Aug_no_death
For 8 samples 14Aug_no_death
For 2 samples 14Aug_low_death
For 3 samples 14Aug_low_death
For 4 samples 14Aug_low_death
For 5 samples 14Aug_low_death
For 6 samples 14Aug_low_death
For 7 samples 14Aug_low_death
For 8 samples 14Aug_low_death
For 2 samples 14Aug_mid_death
For 3 samples 14Aug_mid_death
For 4 samples 14Aug_mid_death
For 5 samples 14Aug_mid_death
For 6 samples 14Aug_mid_death
For 7 samples 14Aug_mid_death
For 8 samples 14Aug_mid_death
For 2 samples 14Aug_high_death
For 3 samples 14Aug_high_death
For 4 samples 14Aug_high_death
For 5 samples 14Aug_high_death
For 6 samples 14Aug_high_death
For 7 samples 14Aug_high_death
For 8 samples 14Aug_high_death
For 2 samples 14Aug_vhigh_death
For 3 samples 14Aug_vhigh_death
For 4 samples 14Aug_vhigh_death
For 5 samples 14Aug_vhigh_death
For 6 samples 14Aug_vhigh_death


In [9]:
filenames = ["14Aug_0.4", "14Aug_0.6"]
mut_rates = [0.4, 0.6]
matrix_names = ["model_3/"+filename for filename in filenames]


batches = len(selections)
batch_size = 5

for (filename, mut_rate, matrix_name) in zip(filenames, mut_rates, matrix_names):
    for no_samples in range(2, 9):
        print("For "+str(no_samples) + " samples", filename)
        stat_matrix = []
        params_per_sim = []
        for batch in range(batches):
            selection_level = selections[batch] #selection per batch
            for number in range(batch_size*batch+1, batch_size*(batch+1)+1): 
                try:
                    stats = calculate_output_stats(matrix_name, base_filepath, number, no_samples)
                    stat_matrix.append(stats)
                    params_per_sim.append([selection_level, mut_rate])
                    #print(stats)
                except:
                    print(filename, matrix_name, number, "failed")
            #print(batch, " complete")
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_stats.npy", stat_matrix)
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_params.npy", params_per_sim)
        

For 2 samples 14Aug_0.4
For 3 samples 14Aug_0.4
For 4 samples 14Aug_0.4
For 5 samples 14Aug_0.4
For 6 samples 14Aug_0.4
For 7 samples 14Aug_0.4
For 8 samples 14Aug_0.4
For 2 samples 14Aug_0.6
14Aug_0.6 model_3/14Aug_0.6 300 failed
14Aug_0.6 model_3/14Aug_0.6 305 failed
14Aug_0.6 model_3/14Aug_0.6 525 failed
14Aug_0.6 model_3/14Aug_0.6 1270 failed
14Aug_0.6 model_3/14Aug_0.6 2145 failed
14Aug_0.6 model_3/14Aug_0.6 7970 failed
14Aug_0.6 model_3/14Aug_0.6 9885 failed
For 3 samples 14Aug_0.6
14Aug_0.6 model_3/14Aug_0.6 300 failed
14Aug_0.6 model_3/14Aug_0.6 305 failed
14Aug_0.6 model_3/14Aug_0.6 525 failed
14Aug_0.6 model_3/14Aug_0.6 1270 failed
14Aug_0.6 model_3/14Aug_0.6 2145 failed
14Aug_0.6 model_3/14Aug_0.6 7970 failed
14Aug_0.6 model_3/14Aug_0.6 9885 failed
For 4 samples 14Aug_0.6
14Aug_0.6 model_3/14Aug_0.6 300 failed
14Aug_0.6 model_3/14Aug_0.6 305 failed
14Aug_0.6 model_3/14Aug_0.6 525 failed
14Aug_0.6 model_3/14Aug_0.6 1270 failed
14Aug_0.6 model_3/14Aug_0.6 2145 failed
14Aug_0.6

In [10]:
filenames = ["13Aug_no_death", "13Aug_low_death", "13Aug_mid_death", "13Aug_high_death", "13Aug_vhigh_death", "13Aug_highest_death"]
death_rates = [0.0, 0.1, 0.2, 0.3, 0.35, 0.38]
matrix_names = ["model_4/"+filename for filename in filenames]


batches = len(selections)
batch_size = 5

for (filename, death_rate, matrix_name) in zip(filenames, death_rates, matrix_names):
    for no_samples in range(2, 9):
        print("For "+str(no_samples) + " samples", filename)
        stat_matrix = []
        params_per_sim = []
        for batch in range(batches):
            selection_level = selections[batch] #selection per batch
            for number in range(batch_size*batch+1, batch_size*(batch+1)+1): 
                try:
                    stats = calculate_output_stats(matrix_name, base_filepath, number, no_samples)
                    stat_matrix.append(stats)
                    params_per_sim.append([selection_level, death_rate])
                    #print(stats)
                except:
                    print(filename, matrix_name, number, "failed")
            #print(batch, " complete")
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_stats.npy", stat_matrix)
        np.save(base_filepath+"/"+matrix_name+"_"+str(no_samples)+"_samples_params.npy", params_per_sim)
        

For 2 samples 13Aug_no_death
For 3 samples 13Aug_no_death
For 4 samples 13Aug_no_death
For 5 samples 13Aug_no_death
For 6 samples 13Aug_no_death
For 7 samples 13Aug_no_death
For 8 samples 13Aug_no_death
For 2 samples 13Aug_low_death
For 3 samples 13Aug_low_death
For 4 samples 13Aug_low_death
For 5 samples 13Aug_low_death
For 6 samples 13Aug_low_death
For 7 samples 13Aug_low_death
For 8 samples 13Aug_low_death
For 2 samples 13Aug_mid_death
For 3 samples 13Aug_mid_death
For 4 samples 13Aug_mid_death
For 5 samples 13Aug_mid_death
For 6 samples 13Aug_mid_death
For 7 samples 13Aug_mid_death
For 8 samples 13Aug_mid_death
For 2 samples 13Aug_high_death
For 3 samples 13Aug_high_death
For 4 samples 13Aug_high_death
For 5 samples 13Aug_high_death
For 6 samples 13Aug_high_death
For 7 samples 13Aug_high_death
For 8 samples 13Aug_high_death
For 2 samples 13Aug_vhigh_death
For 3 samples 13Aug_vhigh_death
For 4 samples 13Aug_vhigh_death
For 5 samples 13Aug_vhigh_death
For 6 samples 13Aug_vhigh_death
